# 8. Counting statistics against the exact tilted Liouvillian — Table V (Sec. VI.B)

The counting-field-tilted generator only raises the label in the displaced frame, so it is upper triangular: the spectrum is $\Lambda^\chi_{mn}=\Lambda_{mn}+\kappa|\alpha|^2(e^\chi-1)$, the propagator is unchanged, and the left eigenoperator $Y_{00}$ is the resummed ladder. The interacting corrections are the same diagrams with $Y_{00}$ as external operator; the tilted eigenvalue $\theta(\chi)$ follows from the full Rayleigh–Schrödinger recursion, Eq. (thetaN), including the renormalization terms $-\sum_k\theta_k\psi_{N-k}$. All cumulants are derivatives of $\theta(\chi)$, taken by a Cauchy contour integral in the complex $\chi$ plane (finite differences carry a bias at $h=0.05$).

References: mean field (Poissonian), the Gaussian/HFB model with its own tilted statistics, and the exact tilted Liouvillian, converged in the fluctuation cutoff. Every tilted calculation is checked against $c_1=\kappa\langle a^\dagger a\rangle$.

**Part A** runs the single-cavity and $K=3$ calculations here; **Part B** regenerates Table V from `benchmarks/truncation/` (`table5_contour.py`, `fcs_rs.py`, `fcs_nz.py`, `k1_radius.py`; minutes each) and compares the generated LaTeX table with the one printed in the paper, number by number (from the LaTeX sources if `PAPER_DIR` points to them, otherwise from the snapshot in `notebooks/paper_reference/`).

In [1]:
from common import *
FAST = True      # True: reduced orders and cutoffs, minutes.  False: the orders and cutoffs of the paper (see the runtime note)
print("repository:", os.path.basename(CODE), "| FAST =", FAST, "| paper sources:", "found" if PAPER else "not present (Tables V and SI from notebooks/paper_reference/)")

repository: gh_work | FAST = True | paper sources: not present (Tables V and SI from notebooks/paper_reference/)


## Part A — the calculation: single cavity ($K=1$) and the three-site ring

In [2]:
from fcs import theta_series
from tilted_exact import theta_lead, cumulants
from gaussian_fcs import gaussian_model
kappa, Dc, eta = 1.0, -1.0, 1.0
for U in [0.05,0.10]:
    Ncf=26 if not FAST else 20
    A_=destroy(Ncf); Ad_=A_.conj().T
    Hx=Dc*Ad_@A_+1j*(eta*Ad_-np.conj(eta)*A_)+U/2*Ad_@Ad_@A_@A_
    ce=cumulants(lambda x: theta_lead(Hx,[A_],[kappa],[x]))
    rho=steady(Hx,[np.sqrt(kappa)*A_]); nss=np.trace(Ad_@A_@rho).real
    Hg,ag,_,p=gaussian_model(kappa,Dc,eta,U,Nc=Ncf)
    cg=cumulants(lambda x: theta_lead(Hg,[ag],[kappa],[x]))
    cd=cumulants(lambda x: np.cumsum(theta_series(kappa,Dc,eta,U,x,6 if FAST else 8)).real[-1])
    cmf=(kappa*abs(p['alpha'])**2,1.0,1.0)
    print(f"U={U}:  c1 identity check: exact c1 {ce[0]:.5f} vs kappa*n {kappa*nss:.5f}")
    for name,c in [("exact",ce),("mean field",cmf),("Gaussian FCS",cg),("diagrams",cd)]:
        print(f"   {name:<14} c1 {c[0]:8.5f}  Fano {c[1]:8.5f}  c3/c1 {c[2]:8.5f}")
    print()
print("(these are finite-difference cumulants at h = 0.05; Table V uses the Cauchy contour, which removes the finite-difference bias — compare Part B)")
from fcs_multi import TiltedChain, cumulants as cum_multi
det3=[-1.0]*3; eta3=np.array([1.0,0.0,0.0])
def build(chi,U=0.05):
    ch=TiltedChain(3,1.0,det3,0.4,eta3,U,ring=True); ch.set_tilt(0,chi,1.0); return ch
for N,md_ in [(2,4),(3,5)] if FAST else [(2,4),(3,5),(4,6)]:
    c=cum_multi(build,None,N,md_,h=0.05)
    print(f"K=3 diagrams N<={N}: c1 {c[0]:.5f}  Fano {c[1]:.5f}  c3/c1 {c[2]:.5f}")

U=0.05:  c1 identity check: exact c1 0.86004 vs kappa*n 0.85959
   exact          c1  0.86004  Fano  1.07466  c3/c1  1.25233
   mean field     c1  0.85869  Fano  1.00000  c3/c1  1.00000
   Gaussian FCS   c1  0.86002  Fano  1.07281  c3/c1  1.23049
   diagrams       c1  0.86004  Fano  1.07458  c3/c1  1.25128



U=0.1:  c1 identity check: exact c1 0.94691 vs kappa*n 0.94597
   exact          c1  0.94691  Fano  1.27100  c3/c1  2.43100
   mean field     c1  0.93994  Fano  1.00000  c3/c1  1.00000
   Gaussian FCS   c1  0.94603  Fano  1.22900  c3/c1  1.77899
   diagrams       c1  0.94612  Fano  1.25245  c3/c1  2.06575

(these are finite-difference cumulants at h = 0.05; Table V uses the Cauchy contour, which removes the finite-difference bias — compare Part B)
K=3 diagrams N<=2: c1 1.14286  Fano 1.13830  c3/c1 1.45888


K=3 diagrams N<=3: c1 1.14574  Fano 1.15559  c3/c1 1.55464


## Part B — Table V of the paper

`assemble_truncation.py` writes `table5_corrected.json` and the LaTeX body `table5_fragment.tex` from the contour-integral references and the stored diagram series; the cell compares that body with the table as printed in the paper.

In [3]:
run("python3 assemble_truncation.py", "benchmarks/truncation", timeout=600)
print(open(os.path.join(BENCH, "truncation", "table5_fragment.tex")).read())
def rows_of(txt):
    m = re.search(r"\\hline(.*?)\\end\{tabular\}", txt[txt.find("tab:fcs"):] if "tab:fcs" in txt else txt, re.S)
    return [re.findall(r"-?\d+\.\d+(?:\\times10\^\{-?\d+\})?", l) for l in m.group(1).split("\\\\") if "&" in l]
tex, where = paper_table("tab:fcs")
A = rows_of(tex); B = rows_of(open(os.path.join(BENCH, "truncation", "table5_fragment.tex")).read())
print(f"Table V as printed ({where}) equals the generated table, number by number:", A == B, f"({len(A)} rows)")

$ (cd code/benchmarks/truncation; python3 assemble_truncation.py)   [11.8 s, exit 0]
 }
}
assembled
\begin{tabular}{llccc}
 & method & $c_1/\kappa$ & $c_2/c_1$ & $c_3/c_1$ \\ \hline
$K=1$, $U=0.05\kappa$   & exact                             & $0.8596$ & $1.0749$ & $1.2511$ \\
                        & mean field                        & $0.8576$ & $1.0000$ & $1.0000$ \\
                        & Gaussian FCS                      & $0.8596$ & $1.0730$ & $1.2296$ \\
                        & diagrams $N\leq8$                 & $0.8596$ & $1.0749$ & $1.2510$ \\[2pt]
$K=1$, $U=0.10\kappa$   & exact                             & $0.9460$ & $1.2702$ & $2.3585$ \\
                        & mean field                        & $0.9327$ & $1.0000$ & $1.0000$ \\
                        & Gaussian FCS                      & $0.9453$ & $1.2292$ & $1.7759$ \\
                        & diagrams $N\leq8$                 & $0.9458$ & $1.2629$ & $2.1848$ \\[2pt]
$K=3$, $U=0.05\kappa$   & exact         

In [4]:
T5 = J("benchmarks", "truncation", "table5_corrected.json")
print("sensitivity of Table V to the reference method (in parentheses: finite differences at h = 0.05 in the laboratory basis)")
quoted("K=1, U=0.10: exact c3/c1 (finite differences: 2.4310)", 2.3585, T5["K1"]["0.10"]["exact"]["c3c1"], "{:.4f}")
quoted("K=3, U=0.05: exact c3/c1 (lab basis, finite differences: 1.6116)", 1.6393, T5["K3"]["0.05"]["exact"]["c3c1"], "{:.4f}")
quoted("K=1, U=0.05: mean-field c1 (finite differences: 0.8587)", 0.8576, T5["K1"]["0.05"]["mean_field"]["c1"], "{:.4f}")
quoted("K=1, U=0.10: mean-field c1 (finite differences: 0.9399)", 0.9327, T5["K1"]["0.10"]["mean_field"]["c1"], "{:.4f}")
print("\nexact reference, independence of the Fock cutoff at contour radius r = 0.1 (K=1, U=0.10):")
for k, v in T5["K1"]["0.10"]["exact_all"].items():
    if "r0.1_" in k: print(f"   {k:18s} c1 {v['c1']:.10f}  Fano {v['fano']:.10f}  c3/c1 {v['c3c1']:.10f}")
print("   (a radius r = 0.2 encloses the branch point of theta(chi) and gives wrong cumulants: the counting expansion holds for small chi, as stated in the paper; use r <= 0.1)")
print("\nK=3 exact: displaced frame, cutoff uncertainty", {k: f"{v:.1e}" for k, v in T5["K3"]["0.05"]["exact"]["cutoff_uncertainty"].items()})

sensitivity of Table V to the reference method (in parentheses: finite differences at h = 0.05 in the laboratory basis)
  K=1, U=0.10: exact c3/c1 (finite differences: 2.4310)          paper       2.3585   code       2.3585
  K=3, U=0.05: exact c3/c1 (lab basis, finite differences: 1.6116) paper       1.6393   code       1.6393
  K=1, U=0.05: mean-field c1 (finite differences: 0.8587)        paper       0.8576   code       0.8576
  K=1, U=0.10: mean-field c1 (finite differences: 0.9399)        paper       0.9327   code       0.9327

exact reference, independence of the Fock cutoff at contour radius r = 0.1 (K=1, U=0.10):
   cut20_r0.1_M32     c1 0.9459713169  Fano 1.2702255577  c3/c1 2.3584926075
   cut26_r0.1_M32     c1 0.9459713172  Fano 1.2702255811  c3/c1 2.3584941806
   cut32_r0.1_M32     c1 0.9459713172  Fano 1.2702255811  c3/c1 2.3584941806
   (a radius r = 0.2 encloses the branch point of theta(chi) and gives wrong cumulants: the counting expansion holds for small chi, as state